# LKH-3 Solver

Downloads and runs LKH-3 on all benchmark instances. Output uploaded as `elfateh/dana-results-lkh3`.

In [ ]:
import os, subprocess, sys, json, time, csv, re, tempfile

subprocess.run(["pip", "install", "-q", "numpy", "kagglehub", "requests"], check=True)

INSTANCE_DIR = "/kaggle/working/instances"
if not os.path.exists(INSTANCE_DIR):
    try:
        import kagglehub
        path = kagglehub.dataset_download("elfateh/dana-instances")
        subprocess.run(["cp", "-r", os.path.join(path, "*"), "/kaggle/working/instances/"], check=True)
    except Exception as e:
        print(f"Kaggle download failed, trying GitHub: {e}")
        import requests as _requests
        BASE_URL = "https://raw.githubusercontent.com/PyVRP/Instances/main"
        SETS = {
            "cordeau": ("MDVRPTW", [f"PR{i}{s}.vrp" for i in range(11, 25) for s in ("A", "B")]),
            "solomon": ("VRPTW/Solomon", ["C101.vrp","C102.vrp","C103.vrp","C104.vrp","C105.vrp","C106.vrp","C107.vrp","C108.vrp","C109.vrp","C201.vrp","C202.vrp","C203.vrp","C204.vrp","C205.vrp","C206.vrp","C207.vrp","C208.vrp","R101.vrp","R102.vrp","R103.vrp","R104.vrp","R105.vrp","R106.vrp","R107.vrp","R108.vrp","R109.vrp","R110.vrp","R111.vrp","R112.vrp","R201.vrp","R202.vrp","R203.vrp","R204.vrp","R205.vrp","R206.vrp","R207.vrp","R208.vrp","R209.vrp","R210.vrp","R211.vrp","RC101.vrp","RC102.vrp","RC103.vrp","RC104.vrp","RC105.vrp","RC106.vrp","RC107.vrp","RC108.vrp","RC201.vrp","RC202.vrp","RC203.vrp","RC204.vrp","RC205.vrp","RC206.vrp","RC207.vrp","RC208.vrp"]),
            "gehring": ("VRPTW", [f"{t}{n}_10_{i}.vrp" for t in ("C","R","RC") for n in (1,2) for i in range(1,11)]),
            "x_instances": ("CVRP", ["X-n101-k25.vrp","X-n106-k14.vrp","X-n110-k13.vrp","X-n115-k10.vrp","X-n120-k6.vrp","X-n125-k30.vrp","X-n129-k18.vrp","X-n134-k13.vrp","X-n139-k10.vrp","X-n143-k7.vrp","X-n153-k22.vrp","X-n157-k13.vrp","X-n162-k11.vrp","X-n167-k10.vrp","X-n176-k26.vrp","X-n181-k23.vrp","X-n186-k15.vrp","X-n190-k8.vrp","X-n195-k51.vrp","X-n200-k36.vrp","X-n204-k19.vrp","X-n209-k16.vrp","X-n214-k11.vrp","X-n219-k73.vrp","X-n223-k34.vrp","X-n228-k23.vrp","X-n233-k16.vrp","X-n237-k14.vrp","X-n242-k48.vrp","X-n247-k50.vrp"]),
        }
        for sname, (rdir, files) in SETS.items():
            dest = os.path.join(INSTANCE_DIR, sname)
            os.makedirs(dest, exist_ok=True)
            for fname in files:
                url = f"{BASE_URL}/{rdir}/{fname}"
                path_d = os.path.join(dest, fname)
                if not os.path.exists(path_d):
                    try:
                        r = _requests.get(url, timeout=30)
                        if r.status_code == 200:
                            with open(path_d, "w") as f:
                                f.write(r.text)
                    except Exception:
                        pass
        print("Downloaded instances from GitHub")

LKH_BIN = "/usr/local/bin/LKH-3"
if not os.path.exists(LKH_BIN):
    print("Installing LKH-3...")
    subprocess.run(["wget", "-q", "http://webhotel4.ruc.dk/~keld/research/LKH-3/LKH-3.0.9.tgz", "-O", "/tmp/LKH-3.tgz"], check=True)
    subprocess.run(["tar", "-xzf", "/tmp/LKH-3.tgz", "-C", "/tmp"], check=True)
    subprocess.run(["make", "-C", "/tmp/LKH-3.0.9"], check=True)
    subprocess.run(["cp", "/tmp/LKH-3.0.9/LKH", LKH_BIN], check=True)
    print("LKH-3 installed.")


In [ ]:
# No time limit - LKH-3 runs until convergence
NUM_RUNS = 1
SEED = 42
TOUR_FILE = "/tmp/lkh_tour.txt"

BENCHMARKS = {
    "cordeau": {"dir": "cordeau", "problem_type": "mdvrptw"},
    "solomon": {"dir": "solomon", "problem_type": "vrptw"},
    "gehring": {"dir": "gehring", "problem_type": "vrptw"},
    "x_instances": {"dir": "x_instances", "problem_type": "cvrp"},
}

def run_lkh3(instance_file, num_runs=1, seed=42):
    tour_file = TOUR_FILE
    with tempfile.NamedTemporaryFile(mode="w", suffix=".par", delete=False) as par_f:
        par_f.write(f"PROBLEM_FILE = {instance_file}\n")
        par_f.write(f"RUNS = {num_runs}\n")
        par_f.write(f"SEED = {seed}\n")
        # No TIME_LIMIT in .par: LKH-3 runs until convergence
        par_f.write(f"OUTPUT_TOUR_FILE = {tour_file}\n")
        par_path = par_f.name

    if os.path.exists(tour_file):
        os.unlink(tour_file)

    cmd = [LKH_BIN, par_path]
    # Long timeout since no time_limit; LKH-3 converges naturally
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=7200)
    if proc.returncode != 0:
        print(f"  LKH stderr: {proc.stderr[:500]}")

    cost = None
    for line in proc.stdout.split("\n"):
        m = re.search(r"Cost\s*=\s*([0-9.]+)", line)
        if m:
            cost = float(m.group(1))
            break

    routes = []
    if os.path.isfile(tour_file) and os.path.getsize(tour_file) > 0:
        with open(tour_file) as f:
            lines = f.readlines()
        tour = []
        for line in lines:
            line = line.strip()
            if not line:
                continue
            try:
                tour.append(int(line))
            except ValueError:
                continue
        current_route = []
        for node in tour:
            if node == 1:
                if current_route:
                    routes.append(current_route)
                    current_route = []
            else:
                current_route.append(node - 2)
        if current_route:
            routes.append(current_route)

    if os.path.exists(par_path):
        os.unlink(par_path)

    return {"cost": cost, "routes": routes, "status": "success" if cost else "no_solution"}


In [ ]:
OUT_CSV = "/kaggle/working/lkh3_results.csv"
rows = []

for bench_name, bench_info in BENCHMARKS.items():
    bench_dir = os.path.join(INSTANCE_DIR, bench_info["dir"])
    if not os.path.exists(bench_dir):
        print(f"SKIP: {bench_dir} not found")
        continue
    instance_files = sorted([f for f in os.listdir(bench_dir) if f.endswith(".vrp")])
    print(f"\n=== {bench_name} ({len(instance_files)} instances) ===")

    for fname in instance_files:
        inst_file = os.path.join(bench_dir, fname)
        inst_name = fname.replace(".vrp", "")
        t0 = time.time()
        try:
            result = run_lkh3(inst_file, NUM_RUNS, SEED)
            elapsed = time.time() - t0
            rows.append({
                "instance": inst_name,
                "solver": "lkh3",
                "benchmark": bench_name,
                "problem_type": bench_info["problem_type"],
                "cost": result["cost"],
                "raw_euclidean_cost": result["cost"],
                "cost_unit": "euclidean",
                "time_s": round(elapsed, 2),
                "status": result["status"],
                "error_msg": "",
            })
            print(f"  {inst_name}: {result['cost']} ({elapsed:.1f}s)")
        except Exception as e:
            elapsed = time.time() - t0
            rows.append({
                "instance": inst_name,
                "solver": "lkh3",
                "benchmark": bench_name,
                "problem_type": bench_info["problem_type"],
                "cost": None,
                "raw_euclidean_cost": None,
                "cost_unit": "euclidean",
                "time_s": round(elapsed, 2),
                "status": "error",
                "error_msg": str(e),
            })
            print(f"  {inst_name}: ERROR {e}")

with open(OUT_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["instance", "solver", "benchmark", "problem_type",
                                            "cost", "raw_euclidean_cost", "cost_unit", "time_s",
                                            "status", "error_msg"])
    writer.writeheader()
    writer.writerows(rows)

print(f"\nSaved {len(rows)} results to {OUT_CSV}")


In [ ]:
DATASET_DIR = "/kaggle/working/dana-results-lkh3"
os.makedirs(DATASET_DIR, exist_ok=True)
subprocess.run(["cp", OUT_CSV, os.path.join(DATASET_DIR, "lkh3_results.csv")], check=True)

with open(os.path.join(DATASET_DIR, "dataset-metadata.json"), "w") as f:
    json.dump({
        "title": "LKH3 Results",
        "id": "elfateh/dana-results-lkh3",
        "licenses": [{"name": "CC0-1.0"}],
    }, f)

# Always try version first (dataset already exists), fall back to create
r = subprocess.run(
    ["kaggle", "datasets", "version", "-p", DATASET_DIR, "-m", "Updated results", "--dir-mode", "zip"],
    capture_output=True, text=True,
)
if r.returncode != 0:
    r2 = subprocess.run(
        ["kaggle", "datasets", "create", "-p", DATASET_DIR, "--dir-mode", "zip"],
        capture_output=True, text=True,
    )
    if r2.returncode != 0:
        print(f"Dataset upload failed: {r2.stderr.strip()}")
    else:
        print("Dataset created successfully.")
else:
    print("Dataset version updated successfully.")